# 02 · Ground-truth evaluation set
The archive only contains `Images/` (334 frames) and `masks/` (305 masks). The paper's **29 ground-truth frames** are the frames where the damage is not visible, so they have **no mask**. This notebook:

1. extracts those 29 frames (`Images` minus `masks`) and checks the counts,
2. finds the vase in each one,
3. creates the paper's four test masks per frame — *rectangle, marginal, large_marginal, multiple* — reproducibly (seeded),
4. writes a contact sheet so you can check every mask by eye.

CPU only — no GPU needed. Resumable.

In [ ]:
# --- Bootstrap: mount Drive, get the code, make `heritage` importable ---
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/abhishekr1403/Artifact_restoration_Project_working"  # <- set after you push to GitHub
REPO_DIR = Path("/content/artefact-restoration-3d")
DRIVE_CODE = Path("/content/drive/MyDrive/artefact-restoration-3d/code")   # fallback: upload the repo folder here

from google.colab import drive
drive.mount("/content/drive")

if "YOUR_USERNAME" not in REPO_URL:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif DRIVE_CODE.exists():
    # Re-sync from Drive on EVERY run so an updated code folder is never ignored
    shutil.copytree(DRIVE_CODE, REPO_DIR, dirs_exist_ok=True)
else:
    raise SystemExit("Set REPO_URL (after pushing to GitHub) or upload the repo folder to " + str(DRIVE_CODE))

sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
print("Repo ready at", REPO_DIR)


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import subprocess, sys
from heritage import paths
cfg = paths.load_config(); P = paths.get_paths(cfg).ensure()
# The dataset lives on the Colab VM disk (not Drive), so a new session needs it again.
# The downloader skips anything already present, so this is safe to run every time.
subprocess.run([sys.executable, 'scripts/download_dataset.py'], check=True)
assert (P.data / cfg['dataset']['images_subdir']).exists(), 'Dataset folder not found after download'

### Quick look at the annotated masks
Checks the mask file format (white-on-black or black-on-white) and how much of the frame the annotators covered.

In [ ]:
import cv2, numpy as np
from heritage.ids import list_images
from heritage.synth_masks import to_bool_mask
mdir = P.data / cfg['dataset']['masks_subdir']
files = list_images(mdir)
for f in files[:: len(files)//6][:6]:
    raw = cv2.imread(str(f), cv2.IMREAD_UNCHANGED)
    b = to_bool_mask(raw)
    print(f.name, raw.shape, raw.dtype, 'unique values:', np.unique(raw)[:4], f'-> covers {b.mean():.1%} of frame')

In [ ]:
!python scripts/prepare_gt.py

### Check the result
Columns: **original · vase estimate (orange) · rectangle · marginal · large_marginal · multiple**.
Every red region should lie on the vase, and the orange outline should follow the vase.

In [ ]:
from PIL import Image
out = P.stage('gt')
sheet = Image.open(out / 'previews' / 'gt_masks_contact_sheet.jpg')
print(sheet.size)
# show in three chunks so each row is readable
w, h = sheet.size; third = h // 3 // 144 * 144
for i in range(0, h, third):
    display(sheet.crop((0, i, w, min(h, i + third))))

In [ ]:
import json
man = json.loads((out / 'manifest.json').read_text())
print('GT frames :', man['n_gt'], man['gt_runs'])
print('damaged   :', man['n_damaged'], man['damaged_runs'])
rep = json.loads((out / 'mask_report.json').read_text())
bad = [r['frame'] for r in rep if not 0.05 < r['vase_frac_of_image'] < 0.8]
print('frames whose vase estimate looks suspicious:', bad or 'none')

### If something looks wrong
| Symptom | Fix |
|---|---|
| `ValueError: expected 29 GT frames` | The split rule found a different number. Send me the printed message — do not edit the numbers in the config. |
| Orange vase outline is off for some frames | Re-run with `!python scripts/prepare_gt.py --no-prior` after deleting `results/gt/vase_masks` and `results/gt/masks`, and compare. Send me the sheet. |
| Masks missing/odd after an interrupted run | Just re-run the cell; finished frames are kept, unfinished ones are redone. |

**Send me** `results/gt/previews/gt_masks_contact_sheet.jpg`, `manifest.json` and the output of the first checks above. I need to confirm the vase estimate before any inpainting is run on these masks.